Examen Practico

Wilson Andres Suarez Mantilla - U00173315


In [ ]:
#Importamos las librerias necesarias

import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import os
import zipfile
import requests
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import random
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Montar Google Drive (para guardar el modelo)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Descargar Stanford Dogs Dataset
print("📥 Descargando Stanford Dogs Dataset...")

# URL del dataset
dataset_url = "http://vision.stanford.edu/aditya86/ImageNetDogs/images.tar"

# Descargar usando wget
!wget -O /content/dogs_dataset.tar http://vision.stanford.edu/aditya86/ImageNetDogs/images.tar

print("✅ Descarga completada!")

In [ ]:
# Extraer el archivo descargado

# Extraer el tar
!tar -xf /content/dogs_dataset.tar -C /content/

# Verificar que se extrajo correctamente
!ls /content/Images/ | head -20  # Muestra las primeras 20 carpetas (razas)

In [ ]:
# Aplicamos la (técnica boxing)
def extract_dog_region_boxing(image_path, output_path, target_size=(150, 150)):
    try:
        # Cargar imagen
        img = cv2.imread(image_path)
        if img is None:
            return False

        # Convertir a RGB
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Convertir a escala de grises para detección de bordes
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # Mejorar contraste con CLAHE (ayuda a detectar mejor al perro)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        gray = clahe.apply(gray)

        # Detectar bordes con Canny
        edges = cv2.Canny(gray, 50, 150)

        # Dilatar bordes para conectar regiones
        kernel = np.ones((5,5), np.uint8)
        edges = cv2.dilate(edges, kernel, iterations=1)

        # Encontrar contornos
        contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if contours:
            # Encontrar el contorno más grande (asumimos que es el perro)
            largest_contour = max(contours, key=cv2.contourArea)
            x, y, w, h = cv2.boundingRect(largest_contour)

            # Extraer solo la región del perro (BOXING)
            dog_region = img_rgb[y:y+h, x:x+w]

            # Redimensionar al tamaño objetivo (menor de 200 como pidió el profesor)
            dog_region_resized = cv2.resize(dog_region, target_size)

            # Guardar la imagen procesada
            cv2.imwrite(output_path, cv2.cvtColor(dog_region_resized, cv2.COLOR_RGB2BGR))
            return True

    except Exception as e:
        pass

    # Si falla, guardar la imagen original redimensionada
    try:
        img = cv2.imread(image_path)
        img_resized = cv2.resize(img, target_size)
        cv2.imwrite(output_path, img_resized)
        return False
    except:
        return False

print("✅ Función de boxing definida")

In [ ]:
# Función boxing
def extract_dog_region_boxing(image_path, output_path, target_size=(100, 100)):
    try:
        img = cv2.imread(image_path)
        if img is None:
            return False

        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # Mejorar contraste
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        gray = clahe.apply(gray)

        # Detectar bordes
        edges = cv2.Canny(gray, 50, 150)
        kernel = np.ones((5,5), np.uint8)
        edges = cv2.dilate(edges, kernel, iterations=1)

        # Encontrar contornos
        contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if contours:
            # Encontrar el contorno más grande (asumimos que es el perro)
            largest_contour = max(contours, key=cv2.contourArea)
            x, y, w, h = cv2.boundingRect(largest_contour)

            # Extraer solo la región del perro (BOXING)
            dog_region = img_rgb[y:y+h, x:x+w]

            # Redimensionar a 100x100 (más rápido y cumple con <200)
            dog_region_resized = cv2.resize(dog_region, target_size)

            # Guardar la imagen procesada
            cv2.imwrite(output_path, cv2.cvtColor(dog_region_resized, cv2.COLOR_RGB2BGR))
            return True

    except Exception as e:
        pass

    # Fallback: redimensionar imagen original
    try:
        img = cv2.imread(image_path)
        img_resized = cv2.resize(img, target_size)
        cv2.imwrite(output_path, img_resized)
        return False
    except:
        return False

print("✅ Boxing optimizado con tamaño 100x100")

In [ ]:
# Función de preprocesamiento
def preprocess_dataset_with_boxing(input_dir, output_dir):

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Obtener todas las imágenes
    image_paths = []
    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                image_paths.append(os.path.join(root, file))

    success_count = 0
    for img_path in tqdm(image_paths):
        rel_path = os.path.relpath(img_path, input_dir)
        output_path = os.path.join(output_dir, rel_path)
        os.makedirs(os.path.dirname(output_path), exist_ok=True)

        if extract_dog_region_boxing(img_path, output_path):
            success_count += 1

    print(f"✅ Procesamiento completado! {success_count}/{len(image_paths)} imágenes")
    return success_count


In [ ]:
# Verificar imágenes procesadas
total_images = 0
for root, dirs, files in os.walk(output_dir):
    for file in files:
        if file.endswith(('.jpg', '.jpeg', '.png')):
            total_images += 1

print(f"Total de imágenes procesadas: {total_images}")

import random
breeds = [d for d in os.listdir(output_dir) if os.path.isdir(os.path.join(output_dir, d))]

if breeds:
    sample_breed = random.choice(breeds)
    breed_path = os.path.join(output_dir, sample_breed)
    images = [f for f in os.listdir(breed_path) if f.endswith('.jpg')]

    if images:
        img = cv2.imread(os.path.join(breed_path, images[0]))
        h, w = img.shape[:2]
        print(f"\Verificación de tamaño:")
        print(f"   Raza: {sample_breed[:30]}")
        print(f"   Dimensiones: {w} x {h} píxeles")

        plt.figure(figsize=(5,5))
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.title(f"Ejemplo procesado: {sample_breed[:30]}")
        plt.axis('off')
        plt.show()

In [ ]:
# Dividimos los datos en train (80%), validation (10%), test (10%)

import shutil
from sklearn.model_selection import train_test_split

# Directorios
data_dir = "/content/data_split"
train_dir = os.path.join(data_dir, 'train')
val_dir = os.path.join(data_dir, 'val')
test_dir = os.path.join(data_dir, 'test')

# Crear directorios
for d in [train_dir, val_dir, test_dir]:
    os.makedirs(d, exist_ok=True)

# Obtener todas las razas
processed_dir = "/content/Images_Processed"
breeds = [d for d in os.listdir(processed_dir)
          if os.path.isdir(os.path.join(processed_dir, d))]

print(f"Procesando {len(breeds)} razas...")

# Dividir imágenes por raza
for breed in tqdm(breeds):
    # Crear carpetas para esta raza
    os.makedirs(os.path.join(train_dir, breed), exist_ok=True)
    os.makedirs(os.path.join(val_dir, breed), exist_ok=True)
    os.makedirs(os.path.join(test_dir, breed), exist_ok=True)

    # Obtener todas las imágenes de esta raza
    breed_path = os.path.join(processed_dir, breed)
    images = [f for f in os.listdir(breed_path)
             if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    if len(images) > 0:
        # Dividir: 80% train, 20% temp
        train_imgs, temp_imgs = train_test_split(images, train_size=0.8, random_state=42)
        # Dividir temp: 50% val, 50% test (10% cada uno del total)
        val_imgs, test_imgs = train_test_split(temp_imgs, train_size=0.5, random_state=42)

        # Copiar imágenes
        for img in train_imgs:
            shutil.copy2(os.path.join(breed_path, img), os.path.join(train_dir, breed, img))
        for img in val_imgs:
            shutil.copy2(os.path.join(breed_path, img), os.path.join(val_dir, breed, img))
        for img in test_imgs:
            shutil.copy2(os.path.join(breed_path, img), os.path.join(test_dir, breed, img))

# Verificar distribución
print("\nDistribución final:")
print(f"Train: {sum([len(os.listdir(os.path.join(train_dir, b))) for b in breeds])} imágenes")
print(f"Validation: {sum([len(os.listdir(os.path.join(val_dir, b))) for b in breeds])} imágenes")
print(f"Test: {sum([len(os.listdir(os.path.join(test_dir, b))) for b in breeds])} imágenes")

In [ ]:
# Generadores SIN augmentation (el augmentation va DENTRO de la red)
print("📊 Creando generadores (sin augmentation, solo dentro de la red)")
print("="*50)

from tensorflow.keras.preprocessing.image import ImageDataGenerator

# ¡IMPORTANTE! Generador sin augmentation (vacío)
train_datagen = ImageDataGenerator()  # ← Sin rotation, shift, zoom, etc.
val_datagen = ImageDataGenerator()
test_datagen = ImageDataGenerator()

# Crear generadores
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(100, 100),
    batch_size=32,
    class_mode='categorical',
    shuffle=True
)

validation_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(100, 100),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(100, 100),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

num_classes = len(train_generator.class_indices)
print(f"\nClases: {num_classes}")
print(f"Train: {train_generator.samples} imágenes")
print(f"Validation: {validation_generator.samples} imágenes")
print(f"Test: {test_generator.samples} imágenes")

In [ ]:
# Modelo CNN optimizado (sin doble augmentation, con GlobalAvgPooling)

from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam

def create_optimized_model(num_classes):
    """
    Modelo CNN optimizado:
    - Data augmentation SOLO dentro de la red
    - GlobalAveragePooling2D (reduce parámetros drásticamente)
    - Menos de 40M parámetros
    """

    model = models.Sequential([
        # ========== PREPROCESAMIENTO ==========
        layers.Resizing(100, 100, input_shape=(100, 100, 3)),
        layers.Rescaling(1./255),

        # ========== DATA AUGMENTATION (SOLO AQUÍ, UNA VEZ) ==========
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.15),
        layers.RandomZoom(0.15),
        layers.RandomContrast(0.1),

        # ========== 1ra CAPA CONVOLUCIONAL ==========
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # ========== 2da CAPA CONVOLUCIONAL ==========
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # ========== GLOBAL AVERAGE POOLING (en lugar de Flatten) ==========
        layers.GlobalAveragePooling2D(),

        # ========== CAPAS DENSAS ==========
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),

        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        # Capa de salida
        layers.Dense(num_classes, activation='softmax')
    ])

    return model

# Crear modelo
model = create_optimized_model(num_classes)

# Compilar con ADAM (mejor que SGD)
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Mostrar arquitectura
model.summary()

# Verificar parámetros
total_params = model.count_params()
print(f"\n📊 Total parámetros: {total_params:,}")

In [ ]:
# Callbacks para optimizar entrenamiento
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint

# Callbacks
callbacks = [
    # Reduce learning rate si no mejora val_accuracy
    ReduceLROnPlateau(
        monitor='val_accuracy',
        patience=3,
        factor=0.5,
        verbose=1,
        min_lr=1e-6
    ),
    # Early stopping si no mejora por 8 épocas
    EarlyStopping(
        monitor='val_accuracy',
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    # Guardar mejor modelo
    ModelCheckpoint(
        '/content/best_optimized_model.h5',
        monitor='val_accuracy',
        mode='max',
        save_best_only=True,
        verbose=1
    )
]

print("Callbacks configurados:")
print("   - ReduceLROnPlateau: reduce LR si accuracy no mejora por 3 épocas")
print("   - EarlyStopping: detiene si no mejora por 8 épocas")
print("   - ModelCheckpoint: guarda el mejor modelo")

In [ ]:
# Entrenar
history = model.fit(
    train_generator,
    epochs=35,
    validation_data=validation_generator,
    callbacks=callbacks,
    verbose=1
)

print("\n✅ Entrenamiento completado!")

# Evaluar
print("\n📈 EVALUANDO MODELO OPTIMIZADO")
print("="*50)

# Cargar mejor modelo
from tensorflow.keras.models import load_model
best_model = load_model('/content/best_optimized_model.h5')

# Evaluar en test
test_loss, test_accuracy = best_model.evaluate(test_generator, verbose=1)

print(f"\n{'='*60}")
print(f"🎯 RESULTADO FINAL - MODELO OPTIMIZADO")
print(f"{'='*60}")
print(f"Test Accuracy: {test_accuracy*100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")
print(f"Total parámetros: {model.count_params():,}")
print(f"{'='*60}")

random_acc = 1/num_classes
print(f"\n📊 Comparación:")
print(f"   Azar: {random_acc*100:.2f}%")
print(f"   Modelo: {test_accuracy*100:.2f}%")
print(f"   Mejora vs azar: +{(test_accuracy - random_acc)*100:.2f}%")

In [ ]:
# CELDA: Visualizar curvas de aprendizaje
print("📈 Analizando evolución del entrenamiento...")
print("="*50)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfica 1: Pérdida (Loss)
axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_title('Evolución de la Pérdida (Loss)', fontsize=14)
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Gráfica 2: Precisión (Accuracy)
axes[1].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[1].axhline(y=0.0706, color='g', linestyle='--', label=f'Tu resultado (7.06%)', linewidth=2)
axes[1].axhline(y=0.0083, color='r', linestyle='--', label=f'Azar (0.83%)', linewidth=2)
axes[1].set_title('Evolución de la Precisión (Accuracy)', fontsize=14)
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Análisis de tendencia
print("\n📊 Análisis de tendencia:")
print(f"   Últimas 5 épocas - Train Accuracy: {history.history['accuracy'][-5:]}")
print(f"   Últimas 5 épocas - Val Accuracy: {history.history['val_accuracy'][-5:]}")

# Calcular pendiente de las últimas épocas
val_acc_last = history.history['val_accuracy'][-10:]
if len(val_acc_last) > 5:
    trend = (val_acc_last[-1] - val_acc_last[0]) / len(val_acc_last)
    print(f"\n📈 Tendencia de las últimas 10 épocas: {trend*100:.4f}% por época")

    if trend > 0:
        print("   ✅ La curva SIGUE SUBIENDO - Vale la pena entrenar más épocas")
        # Proyección
        epochs_to_70 = (0.70 - history.history['val_accuracy'][-1]) / trend if trend > 0 else float('inf')
        if epochs_to_70 < 500:
            print(f"   📊 Proyección: Para llegar a 70% se necesitarían ~{int(epochs_to_70)} épocas más")
        else:
            print("   ⚠️ Proyección: Para llegar a 70% se necesitarían demasiadas épocas (no es factible)")
    else:
        print("   ⚠️ La curva se ESTANCÓ - Entrenar más épocas no mejorará significativamente")

In [ ]:
# CELDA: Guardar modelo actual
print("💾 Guardando modelo actual como respaldo...")
model.save('/content/modelo_actual_respaldo.h5')
print("✅ Respaldo guardado")